# Baseline B2

Author: Mathias Marryat Husted

This notebook provides a reproducible way to run the baseline B2 runs. Checks are performed at each step to ensure data consistency.

Baseline B2 focuses on fuzzy matching. RapidFuzz has been chosen to facilitate the fuzzy string matching.

First of all, the feasibility of the fuzzy matching will need to be checked. The checks below are to get a feel for how it handles increasingly large files (fit in RAM, etc.) and will inform the decisions made from now on

In [1]:
import pandas as pd
from rapidfuzz import process, fuzz, utils
import gzip, time
import os, psutil

BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
CONTROL_PATH = "../../data/dataset/control.csv"
PUBCHEM_DIR = "../../data/reference"
HGNC_PATH = "../../data/reference/hgnc_complete_set.txt"


benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)
control = pd.read_csv(CONTROL_PATH, dtype=str, keep_default_na=False)


In [12]:
# Check RapidFuzz works
input_row = control["input"]
print("=======GENERAL CHECKS=======")
for input_str in ["myo", "ose", "ate", "min", "NADH"]:
    print(process.extract(input_str, input_row, scorer=fuzz.ratio, processor=None, limit=3))

# Since the abstain rule needs to look at ties, we need to check how many hits share the top score for shorter inputs
print("=======TIES CHECK=======")
hits = process.extract("in", input_row, scorer=fuzz.ratio, processor=None, limit=10)
for hit in hits:
    print(hit)
print("=======NEGATIVE CHECK=======")
print(process.extract("zzzzzzzzzzz", input_row, scorer=fuzz.ratio, processor=None, limit=3))

=======GENERAL CHECKS=======
[('mercury', 40.0, 157), ('myo-inositol', 40.0, 159), ('cytosine', 36.36363636363637, 130)]
[('lactose', 60.0, 152), ('D-ribose', 54.54545454545454, 36), ('cytosine', 54.54545454545454, 130)]
[('lactose', 60.0, 152), ('pyruvate', 54.54545454545454, 172), ('xanthine', 54.54545454545454, 188)]
[('spermine', 54.54545454545454, 179), ('thiamine', 54.54545454545454, 182), ('melatonin', 50.0, 156)]
[('NADH', 100.0, 85), ('NADPH', 88.88888888888889, 86), ('NAD+', 75.0, 84)]
=======TIES CHECK=======
('D-biotin', 40.0, 29)
('cytosine', 40.0, 130)
('spermine', 40.0, 179)
('thiamine', 40.0, 182)
('xanthine', 40.0, 188)
('ibuprofen', 36.36363636363637, 151)
('melatonin', 36.36363636363637, 156)
('serotonin', 36.36363636363637, 174)
('tretinoin', 36.36363636363637, 183)
('L-cysteine', 33.333333333333336, 61)
=======NEGATIVE CHECK=======
[('4-aminobenzoic acid', 6.666666666666665, 7), ('(-)-menthol', 0.0, 0), ('(6R)-5,6,7,8-tetrahydrobiopterin', 0.0, 1)]


### First steps

**1.**

First of all, we need to freeze the configuration we use for RapidFuzz. In this case, 3 things are considered:
* `fuzz.ratio` is used: it compares 2 entire strings and scores similarity by Levenshtein distance (linear runtime)
* `casefold()`: Case-folding will be used as it proved useful in B1
* `limit`: This will be used to assert if the top match is a tie, in which case it should abstain.

**2.**

We need to count the lines of our input and create a similar function to the previous baseline which builds our internal vocabulary from the entire PubChem set.

In [13]:
def pubchem_count_lines(path):
    n_read_lines = 0
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            elements = line.rstrip("\n").split("\t", 1)
            if len(elements) != 2:
                continue
            n_read_lines += 1
    return n_read_lines

def get_pubchem_vocab(path, max_lines=None):
    vocab = {}
    n_read_lines = 0
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            elements = line.rstrip("\n").split("\t", 1)
            if len(elements) != 2:
                continue
            cid, name = elements
            folded = name.casefold()
            if not folded in vocab:
                vocab[folded] = set()
            vocab[folded].add(cid)
            n_read_lines += 1
            if max_lines is not None and n_read_lines >= max_lines:
                break
    return vocab

reduced_vocab = get_pubchem_vocab(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", max_lines=1000000)
print(len(reduced_vocab))

997981


In [14]:
# Check how many lines in total we are working with (This might take long, about 1 min on my system)

#pubchem_n_lines = pubchem_count_lines(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz")
pubchem_n_lines = 117900000 # We don't need to count each time and know the list is about this size

In [15]:
# Base function with a scoring scheme (only returns if there is one single best match)
def get_fuzzy(input_str, vocab, names, limit=15, scorer=fuzz.ratio, score_cutoff=70):
    hits = process.extract(input_str.casefold(), names, limit=limit, scorer=scorer, score_cutoff=score_cutoff, processor=None)
    if len(hits) == 0:
        return {"returned_id": "", "n_candidates": 0, "score": "", "matches": ""}
    best_score = hits[0][1]
    best_names = [hit[0] for hit in hits if hit[1] == best_score]
    cids = []
    for name in best_names:
        for cid in vocab[name]:
            if cid not in cids:
                cids.append(cid)

    # Decision for returned_id
    returned_id = cids[0] if len(cids) == 1 else ""
    return {"returned_id": returned_id, "n_candidates": len(cids), "score": best_score, "matches": " | ".join(best_names)}

names = list(reduced_vocab.keys())
hits = get_fuzzy("glucose", reduced_vocab, names)
print(hits)

{'returned_id': '5793', 'n_candidates': 1, 'score': 100.0, 'matches': 'glucose'}


Glucose is returned as a 100% match after case-folding, as expected

In [16]:
# Take a quick sample from the vocabulary and assert that an entity name always scores 100% on itself 
# Then check the total time needed if we expand this to the control.csv and benchmark.csv files on the whole vocabulary
vocab_sample = []
for item in reduced_vocab.keys():
    vocab_sample.append(item)
    if len(vocab_sample) >= 20:
        break
start_time = time.time()
for name in vocab_sample:
    result = get_fuzzy(name, reduced_vocab, names)
    assert result["score"] == 100
end_time = time.time()
elapsed = end_time - start_time
comp_per_second = (len(vocab_sample)*len(reduced_vocab))/elapsed
print(f"{len(vocab_sample)} queries on {len(reduced_vocab)}-word vocab, elapsed time: {elapsed}")
print(f"Number of comparisons/s: {comp_per_second}")
print(f"Estimated time needed in total for whole data set on whole vocabulary: {((162*pubchem_n_lines)/comp_per_second)/60:.1f} minutes")

20 queries on 997981-word vocab, elapsed time: 1.8180265426635742
Number of comparisons/s: 10978728.600275187
Estimated time needed in total for whole data set on whole vocabulary: 29.0 minutes



We're looking at a runtime of about half an hour in total and a potential exceedance of system memory (16GB) based on the calculation.

Claude was asked to help determine if memory management was going to be an issue and wrote the following function using memory tools to check whether the byte size was going to be an issue when having the whole vocabulary of 117.9M loaded.


In [ ]:
# Claude Sonnet 5.5 wrote this cell (01.10.2026)
# Prompt: Help me determine as simply as possible whether memory will be an issue on 16GB of RAM
# The tool noted the estimate output is an optimistic estimate as longer inputs may increase the average name byte size.
# Inputs:
# - Prompt (as written)
# - This file (current state: file ends above)
# - benchmark.csv
# - control.csv

proc = psutil.Process(os.getpid())
before = proc.memory_info().rss

names = []
with gzip.open(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", "rt", encoding="utf-8") as f:
    for line in f:
        elements = line.rstrip("\n").split("\t", 1)
        if len(elements) != 2:
            continue
        names.append(elements[1].casefold())
        if len(names) == 1_000_000:
            break

after = proc.memory_info().rss
bytes_per_name = (after - before) / len(names)
print(bytes_per_name, "bytes/name")
print(bytes_per_name * pubchem_n_lines / 1e9, "GB for the full list")
print(psutil.virtual_memory().available / 1e9, "GB available")

0.483328 bytes/name
0.056984371199999995 GB for the full list
9.900724224 GB available


Memory is obviously too tight, since the above was a very optimistic estimate. We will have to slice our synonyms file into blocks. If the system starts swapping memory, it will take significantly longer than the 30 minutes indicated above and therefore we want to avoid that and rather slice memory into smaller blocks. In order to avoid failed runs, Claude was therefore asked to help

In [18]:
# This cell was written by Claude Sonnet 5.5, 02.10.2026
# Prompt (summary): merge my get_fuzzy and run_b1_chemicals (adapted to use fuzzy matching as seen in the first cells) so the run function reuses get_fuzzy,
#   keep memory block slicing, reuse as much of my code as possible.
#
# Inputs given to the tool: this notebook (earlier get_fuzzy, run_b1_chemicals from the first baseline, throughput and
#   memory measurements), Ruleset.md (R5.3, B2 configuration), control.csv, benchmark.csv.
#
# Authorship: get_fuzzy, the throughput test and the slicing idea are mine; the tool fixed bugs in
#   get_fuzzy (scorer/processor not passed, query not folded, sets vs lists), drafted run_b2_chemicals,
#   and in this version split the search from the decision and added limit/truncation handling.
# B2 configuration (fixed a priori, mine): fuzz.ratio, processor=None, casefold on query and
#   vocabulary, full CID-Synonym-filtered (no benchmark-derived restriction), no score floor,
#   top score returned, several distinct CIDs at the top score -> abstain, routing by gold entity_type.
#   Feasibility: 5M-line blocks (full list needs ~9.7 GB of 16 GB); limit=100 ties per block per query.


def get_fuzzy(input_str, names, cids, best_score=0.0, limit=100, scorer=fuzz.ratio):
    # Searches ONE block (names and cids are parallel lists, one entry per file line).
    # best_score = best score from earlier blocks; used as cutoff so worse hits are dropped
    hits = process.extract(input_str.casefold(), names, scorer=scorer, processor=None,
                           limit=limit, score_cutoff=best_score)
    if len(hits) == 0:
        return best_score, []
    top_score = hits[0][1]
    pairs = []
    for hit in hits:
        if hit[1] == top_score:
            pairs.append((hit[0], cids[hit[2]]))
    if len(pairs) == limit:
        print(f"WARNING: tie list truncated at {limit} for {input_str!r}")
    return top_score, pairs


def decide(best_score, pairs):
    # Called once per query after all blocks. Counts distinct CIDs, no sets
    names_found = []
    cids_found = []
    for name, cid in pairs:
        if name not in names_found:
            names_found.append(name)
        if cid not in cids_found:
            cids_found.append(cid)
    if len(cids_found) == 0 or best_score == 0:
        return {"returned_id": "", "n_candidates": 0, "score": "", "matched_string": "", "candidate_ids": ""}
    returned_id = cids_found[0] if len(cids_found) == 1 else ""
    return {"returned_id": returned_id, "n_candidates": len(cids_found), "score": best_score,
            "matched_string": "|".join(names_found), "candidate_ids": "|".join(cids_found)}


def score_block(queries, names, cids, best_score, best_pairs):
    for q in queries:
        top_score, pairs = get_fuzzy(q, names, cids, best_score[q])
        if len(pairs) == 0:
            continue
        if top_score > best_score[q]:
            best_score[q] = top_score
            best_pairs[q] = []
        best_pairs[q].extend(pairs)


def run_b2_chemicals(path, queries, block_size=5_000_000, max_lines=None):
    best_score = {}
    best_pairs = {}
    for q in queries:
        best_score[q] = 0.0
        best_pairs[q] = []
    names, cids = [], []
    n_read = 0
    start = time.time()
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            elements = line.rstrip("\n").split("\t", 1)
            if len(elements) != 2:
                continue
            cids.append(elements[0])
            names.append(elements[1].casefold())
            n_read += 1
            if len(names) == block_size:
                score_block(queries, names, cids, best_score, best_pairs)
                names, cids = [], []
                print(f"{n_read} lines, {time.time() - start:.0f} s")
            if max_lines is not None and n_read >= max_lines:
                break
    if len(names) > 0:
        score_block(queries, names, cids, best_score, best_pairs)
    results = {}
    for q in queries:
        results[q] = decide(best_score[q], best_pairs[q])
    return results, n_read

flat_names, flat_cids = [], []
for name in reduced_vocab:
    for cid in reduced_vocab[name]:
        flat_names.append(name)
        flat_cids.append(cid)

# Slice test -> Check that block logic is right
test_queries = vocab_sample + ["asetic acid", "glucosse", "n", "g6p", "acetyl carnitin"]
block_results, _ = run_b2_chemicals(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", test_queries,
                                    block_size=250_000, max_lines=1_000_000)
for q in test_queries:
    top_score, pairs = get_fuzzy(q, flat_names, flat_cids)   # one big block = reference
    ref = decide(top_score, pairs)
    got = block_results[q]
    assert ref["score"] == got["score"], q
    assert sorted(ref["candidate_ids"].split("|")) == sorted(got["candidate_ids"].split("|")), q
print("block logic agrees with the in-memory reference")

250000 lines, 1 s
500000 lines, 1 s
750000 lines, 2 s
1000000 lines, 2 s
block logic agrees with the in-memory reference


In [19]:
benchmark_chemicals = benchmark[benchmark["entity_type"] == "chemical"]
control_chemicals = control[control["entity_type"] == "chemical"]

raw_queries = list(benchmark_chemicals["input"]) + list(control_chemicals["canonical_name"])
queries = []
for q in raw_queries:
    folded = q.casefold()
    if folded not in queries:
        queries.append(folded)
print(len(raw_queries), "raw,", len(queries), "unique folded queries")
# LINES HAVE BEEN COMMENTED OUT AS TO NOT RUN EVERY TIME
#query_results, n_read = run_b2_chemicals(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", queries)
#assert n_read == 117_906_661, n_read

#query_table = pd.DataFrame(query_results).T.reset_index().rename(columns={"index": "query"})
#query_table.to_csv(f"../../results/intermediate/b2_chem_synonym_queries.csv", index=False)   # derived data, not ../results

271 raw, 255 unique folded queries


The data was created (runtime: 40m, 50.2s)

In [20]:
# Now the data is created and saved into a CSV
# We need to merge the resulting query file back into the benchmark
# We read the file produced by the previous step instead of relying on the output variables, as that would require the 30

b2_chem_query = pd.read_csv("../../results/intermediate/b2_chem_synonym_queries.csv", dtype=str, keep_default_na=False)

b2_chem_query.head()


,query,returned_id,n_candidates,score,matched_string,candidate_ids
0,glucosse,5793,1,93.33333333333333,glucose,5793
1,ureaa,1176,1,88.88888888888889,urea,1176
2,hydroxyureaa,3657,1,95.65217391304348,hydroxyurea,3657
3,cytossine,597,1,94.11764705882352,cytosine,597
4,sperrmine,1103,1,94.11764705882352,spermine,1103


In [21]:
benchmark_chemicals.head()

,id,input,clean_form,canonical_name,entity_type,noise_type,noise_detail,id_status,accepted_id,note
0,CHEM-001,glucosse,glucose,D-glucose,chemical,typo,doubled_letter,ambiguous,5793,
1,CHEM-002,ureaa,urea,urea,chemical,typo,doubled_letter,single,1176,
2,CHEM-003,hydroxyureaa,hydroxyurea,hydroxyurea,chemical,typo,doubled_letter,single,3657,Minimal pair with 'ureaa'.
3,CHEM-004,cytossine,cytosine,cytosine,chemical,typo,doubled_letter,single,597,
4,CHEM-005,sperrmine,spermine,spermine,chemical,typo,doubled_letter,single,1103,Adjacent entity: spermidine.


In [22]:
b2_chem_result = []

for row in benchmark_chemicals.itertuples():
    input_match = b2_chem_query[b2_chem_query["query"] == row.input.casefold()]
    assert len(input_match) == 1

    b2_chem_result.append({
        "id": row.id,
        "canonical_name": row.canonical_name,
        "input": row.input,
        "n_candidates": input_match["n_candidates"].iloc[0],
        "noise_type": row.noise_type,
        "id_status": row.id_status,
        "accepted_id": row.accepted_id,
        "returned_id": input_match["returned_id"].iloc[0],
        "score": input_match["score"].iloc[0],
        "matched_string": input_match["matched_string"].iloc[0],
        "entity_type": row.entity_type,
        "candidate_ids": input_match["candidate_ids"].iloc[0],
    })


output_data = pd.DataFrame(b2_chem_result)
output_data.to_csv("../../results/baselines/b2_chem_synonym.csv", index=False)

Now we have created the output file for the chemicals. We now just need `b2_gene_synonym.csv`. First, we need to reconstruct our table from B1 and build the case-folded vocabulary for the fuzzy matching using it.

In [3]:
hgnc_data = pd.read_csv(HGNC_PATH, sep="\t", dtype=str, keep_default_na=False)

FIELD_ORDER = ["symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"]  # Fallback order, extended to names

# One row per (gene, name, field): the gene synonym vocabulary in long format

# The function below was written by Claude Opus 5.5 as described in baseline1_generation.ipynb
def build_gene_synonym_table(hgnc_data):
    approved = hgnc_data[(hgnc_data["status"] == "Approved") & (hgnc_data["symbol"] != "")]
    # Approved symbols are unique, so this is a plain 1:1 lookup
    assert approved["symbol"].is_unique, "duplicate approved symbols in the HGNC data"
    parts = []
    for field in FIELD_ORDER:
        part = approved[["hgnc_id", field]].rename(columns={field: "name"})
        part["name"] = part["name"].str.split("|")   # pipe-separated lists -> Python lists
        part = part.explode("name")                   # one row per name
        part = part[part["name"] != ""]               # genes with an empty field
        part["field"] = field
        parts.append(part)
    table = pd.concat(parts, ignore_index=True)
    table["folded"] = table["name"].str.casefold()
    return table

# Now we will build a vocabulary similar to the chemical vocabulary to fuzzy match against
def build_gene_vocab(synonym_table):
    vocab = {}
    n_read_lines = 0
    for row in synonym_table.itertuples():
        n_read_lines += 1
        if row.folded not in vocab:
            vocab[row.folded] = set()
        vocab[row.folded].add(row.hgnc_id)
    return vocab


gene_table = build_gene_synonym_table(hgnc_data)
gene_vocab = build_gene_vocab(gene_table)
print(f"Length of gene_table: {len(gene_table)}, length of gene_vocab: {len(gene_vocab)}")


Length of gene_table: 199600, length of gene_vocab: 195783


In [24]:
def get_fuzzy_genes(input_str, names, table, limit=100, scorer=fuzz.ratio, score_cutoff=0.0):
    hits = process.extract(input_str.casefold(), names, limit=limit, scorer=scorer, score_cutoff=score_cutoff, processor=None)
    # Get best hit
    if len(hits) == 0:
        return {"returned_id": "", "n_candidates": 0, "score": "", "matched_string": "", "matched_field": "", "candidate_ids": ""}
    
    best_score = hits[0][1]
    best_names = [hit[0] for hit in hits if hit[1] == best_score]
    if len(best_names) == limit:
        print(f"Limit of {limit} exceeded. Input truncated!")
    best_ids = []
    matched_fields = []

    for item in best_names:
        matches = table[table["folded"] == item]
        for match in matches["hgnc_id"]:
            if match not in best_ids:
                best_ids.append(match)

        for field in matches["field"]:
            if field not in matched_fields:
                matched_fields.append(field)

    return {"returned_id": best_ids[0] if len(best_ids) == 1 else "", "n_candidates": len(best_ids), "score": best_score,
            "matched_string": "|".join(best_names), "matched_field": "|".join(matched_fields), "candidate_ids": "|".join(best_ids)}


gene_names = list(gene_vocab.keys())
# Test query
hits = get_fuzzy_genes("tp5", gene_names, gene_table)
hits

{'returned_id': '',
 'n_candidates': 9,
 'score': 85.71428571428572,
 'matched_string': 'rtp5|tp53|atp5|tip5|xtp5|utp5|tp55|tp15|tsp5',
 'matched_field': 'symbol|prev_symbol|alias_symbol',
 'candidate_ids': 'HGNC:26585|HGNC:11998|HGNC:847|HGNC:962|HGNC:29676|HGNC:28945|HGNC:11922|HGNC:1788|HGNC:2227'}

In [25]:
# We re-purpose the function from B1 and adapt it to use RapidFuzz, as with the chemical function.
# The original run_b1_genes function was refined using Claude, as cited in B1.
# All changes are my own.
def run_b2_genes(names, synonym_table, benchmark): # Claude changed this in the previous benchmark: hgnc_data -> synonym_table
    benchmark = benchmark[benchmark["entity_type"] == "gene"]
    # Look up gene data
    results = []
    for row in benchmark.itertuples():

        hits = get_fuzzy_genes(row.input.casefold(), names, synonym_table)
        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": hits["n_candidates"],
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": hits["returned_id"],
            "score": hits["score"],
            "matched_field": hits["matched_field"],
            "matched_string": hits["matched_string"],
            "candidate_ids": hits["candidate_ids"],
        })

    return pd.DataFrame(results)
    

b2_run = run_b2_genes(gene_names, gene_table, benchmark=benchmark)
# Claude Sonnet 5.5 wrote the tests below (03.10.2026)
# Prompt: Help me write a quick test to assert the data integrity of my output
# Inputs:
# - Prompt (as written)
# - The 3 past cells
# - hgnc_complete_set.txt
# - benchmark.csv
# - control.csv
for row in control[control["entity_type"] == "gene"].itertuples():
    res = get_fuzzy_genes(row.canonical_name, gene_names, gene_table)
    assert res["score"] == 100.0, row.id
    assert row.hgnc_id in res["candidate_ids"].split("|"), row.id


assert len(b2_run) == 157 and b2_run["id"].is_unique
for row in b2_run.itertuples():
    assert (row.returned_id != "") == (row.n_candidates == 1), row.id

# End of Claude's code

b2_run.to_csv("../../results/baselines/b3_gene_synonym.csv", index=False)